In [2]:
import tensorflow as tf
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Load data
train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')

# Function to prepare test and train data split
def prepare_data(df, target_col=None):
    df = df.copy()  # To not modify existing data set

    # Separate column types
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns

    # Drop target column if present
    if target_col and target_col in numeric_cols:
        numeric_cols = numeric_cols.drop(target_col)

    # Handle numeric columns
    for col in numeric_cols:  # iterate through numeric
        df[col] = pd.to_numeric(df[col], errors='coerce')  # Convert to numeric
        df[col].fillna(df[col].mean(), inplace=True)  # fillna in place with mean

    # Scale numeric columns
    scaler = StandardScaler()
    df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
    
    # Convert to numpy array and ensure float32 dtype
    return df[numeric_cols].astype('float32')

test_ids = test_data['ID']

# Prepare features
X_train = prepare_data(train_data, target_col='TARGET')
X_test = prepare_data(test_data)

# Prepare target
y_train = train_data['TARGET']

# Create the model
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Compile
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train
print("Training model...")
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=3,
            restore_best_weights=True
        )
    ]
)

# Predict
predictions = model.predict(X_test)

Training model...
Epoch 1/20


/opt/miniconda3/envs/P1/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1901/1901 ━━━━━━━━━━━━━━━━━━━━ 2s 754us/step - accuracy: 0.9563 - loss: 0.1727 - val_accuracy: 0.9566 - val_loss: 0.1591
Epoch 2/20
1901/1901 ━━━━━━━━━━━━━━━━━━━━ 1s 682us/step - accuracy: 0.9606 - loss: 0.1446 - val_accuracy: 0.9566 - val_loss: 0.1561
Epoch 3/20
1901/1901 ━━━━━━━━━━━━━━━━━━━━ 1s 664us/step - accuracy: 0.9608 - loss: 0.1404 - val_accuracy: 0.9567 - val_loss: 0.1553
Epoch 4/20
1901/1901 ━━━━━━━━━━━━━━━━━━━━ 1s 717us/step - accuracy: 0.9626 - loss: 0.1355 - val_accuracy: 0.9567 - val_loss: 0.1556
Epoch 5/20
1901/1901 ━━━━━━━━━━━━━━━━━━━━ 1s 640us/step - accuracy: 0.9615 - loss: 0.1356 - val_accuracy: 0.9566 - val_loss: 0.1549
Epoch 6/20
1901/1901 ━━━━━━━━━━━━━━━━━━━━ 1s 642us/step - accuracy: 0.9613 - loss: 0.1352 - val_accuracy: 0.9565 - val_loss: 0.1535
Epoch 7/20
1901/1901 ━━━━━━━━━━━━━━━━━━━━ 1s 682us/step - accuracy: 0.9614 - loss: 0.1361 - val_accuracy: 0.9566 - val_loss: 0.1543
Epoch 8/20
1901/1901 ━━━━━━━━━━━━━━━━━━━━ 1s 698us/step - accuracy: 0.9613 - loss: 0.13

In [3]:
submission_df = pd.DataFrame({
    'ID': test_ids,
    'TARGET': predictions.flatten()
})
submission_df.to_csv('submission.csv', index=False)
print("Submission file created!")

Submission file created!


In [4]:
train_data.head()

,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
0,1,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39205.170000,0
1,3,2,34,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,49278.030000,0
2,4,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,67333.770000,0
3,8,2,37,0.0,195.0,195.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,64007.970000,0
4,10,2,39,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,117310.979016,0
